# Thermal-cycle plot exporter

This notebook loads every preset from `oven_presets.json` and writes one PNG per saved thermal cycle. The figures use the compact atmosphere strip, temperature profile, debinding/sintering hatching, and cooling region layout.

In [1]:
from pathlib import Path
import json
import re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

import oven_parameters_tool as oven

ROOT = Path.cwd()
PRESETS_FILE = ROOT / 'oven_presets.json'
OUTPUT_DIR = ROOT / 'thermal_cycle_plots'
OUTPUT_DIR.mkdir(exist_ok=True)
GAS_COLORS = oven.GAS_COLORS


In [2]:
def parameters_from_preset(data):
    def steps(key):
        return tuple(oven.ThermalStep(**step) for step in data.get(key, []))
    phases = tuple(oven.GasPhase(**phase) for phase in data.get('gas_phases', []))
    return oven.TemperatureProfileParameters(
        start_temperature_c=data.get('start_temperature_c', 15.0),
        cooling_rate_c_per_h=data.get('cooling_rate_c_per_h', 60.0),
        cycle_mode=data.get('cycle_mode', 'Debinding + sintering'),
        start_time=data.get('start_time', ''),
        debinding_steps=steps('debinding_steps'),
        sintering_steps=steps('sintering_steps'),
        gas_phases=phases,
    )

def safe_name(name):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', name).strip('_') or 'cycle'

with PRESETS_FILE.open(encoding='utf-8') as handle:
    presets = json.load(handle)
list(presets)


['test_n2',
 'test_n1',
 'test_n3',
 'test_n4',
 'Sintering_1',
 'Sintering_2',
 'Debinding_3',
 'Sintering_3',
 'Debinding_&_Sintering_1',
 'Sintering_4',
 'Debinding_&_Sintering_3',
 'Sintering_2.2']

In [3]:
def plot_cycle(name, preset, save=True):
    params = parameters_from_preset(preset)
    times, temps, timing = oven.build_temperature_profile(params)
    times = np.asarray(times)
    temps = np.asarray(temps)
    fig, ax = plt.subplots(figsize=(12, 8), dpi=140)
    fig.subplots_adjust(left=0.10, right=0.78, top=0.78, bottom=0.11)
    ax.set_facecolor('#fffefe')
    peak = max(temps)
    ax.set_xlim(0, timing['total_h'] * 1.02)
    ax.set_ylim(0, max(1250, peak * 1.10))

    # Atmosphere shading and the dedicated top strip.
    gas_intervals = []
    for phase in params.gas_phases:
        color = GAS_COLORS[phase.gas]
        gas_name = {'Nitrogen': 'N$_2$', 'CO2': 'CO$_2$', 'Argon': 'Ar', 'Air': 'Air'}.get(phase.gas, phase.gas)
        display = f"{gas_name}, {phase.flow_rate_l_per_min:g} L/min"
        if phase.gas == 'Air':
            display = 'Air, ambient'
        if phase.basis == 'Whole phase':
            intervals = oven.stage_time_ranges(timing, phase.stage)
        elif phase.basis == 'Time':
            intervals = [(phase.start, phase.end)]
        else:
            intervals = []
            stage_intervals = ([(0.0, timing['sintering_end_h'])]
                              if phase.stage == 'All' else oven.stage_time_ranges(timing, phase.stage))
            for start, end in oven.temperature_range_intervals(times, temps, phase.start, phase.end):
                for stage_start, stage_end in stage_intervals:
                    if min(end, stage_end) > max(start, stage_start):
                        intervals.append((max(start, stage_start), min(end, stage_end)))
        for start, end in intervals:
            ax.axvspan(start, end, color=color, alpha=0.16, zorder=0)
            ax.add_patch(plt.Rectangle((start, 1.0), end-start, 0.12, transform=ax.get_xaxis_transform(),
                                       facecolor=color, edgecolor=color, alpha=0.55, clip_on=False, zorder=5))
            gas_intervals.append((start, end, display, color))

    shown = set()
    for start, end, display, color in gas_intervals:
        key = (display, round(start, 6), round(end, 6))
        if key in shown:
            continue
        shown.add(key)
        ax.text((start + end) / 2, 1.055, display, transform=ax.get_xaxis_transform(),
                ha='center', va='bottom', fontsize=10, fontweight='bold', color=color,
                bbox={'facecolor': color, 'alpha': 0.16, 'edgecolor': color, 'pad': 5}, clip_on=False)
    if gas_intervals:
        ax.text(0.5, 1.18, 'Atmosphere', transform=ax.transAxes, ha='center', va='bottom',
                fontsize=16, fontweight='bold', clip_on=False)
        ax.text(-0.02, 1.055, 'Gas:', transform=ax.transAxes, ha='right', va='bottom',
                fontsize=10, fontweight='bold', clip_on=False)

    # Reference regions and temperature profile.
    for low, high in (oven.PYROLYSIS_RANGE_C, oven.COPPER_SINTERING_RANGE_C):
        intervals = oven.temperature_range_intervals(times, temps, low, high)[:1]
        color = oven.PYROLYSIS_RANGE_COLOR if (low, high) == oven.PYROLYSIS_RANGE_C else oven.COPPER_RANGE_COLOR
        for start, end in intervals:
            ax.axvspan(start, end, facecolor='none', edgecolor=color, hatch='//', linewidth=1.0, alpha=0.65)
            ax.axvline(start, color=color, linestyle='--', linewidth=0.9, alpha=0.8)
            ax.axvline(end, color=color, linestyle='--', linewidth=0.9, alpha=0.8)
    ax.axvspan(timing['sintering_end_h'], timing['total_h'], color='#75AADB', alpha=0.16, zorder=0)
    ax.plot(times, temps, color='#171717', linewidth=2.2, label='Oven temperature', zorder=4)
    ax.axhline(1083, color='#C95B12', linewidth=1.7)
    ax.axhline(750, color='#C95B12', linewidth=1.1, linestyle='--')
    ax.text(0.99, 1083, 'Cu melting (1083 C)', transform=ax.get_yaxis_transform(), ha='right', va='bottom', color='#C95B12', fontweight='bold')
    ax.text(0.99, 750, '750 C', transform=ax.get_yaxis_transform(), ha='right', va='bottom', color='#C95B12', fontweight='bold')
    pyro_intervals = oven.temperature_range_intervals(times, temps, *oven.PYROLYSIS_RANGE_C)[:1]
    if pyro_intervals:
        start, end = pyro_intervals[0]
        ax.text((start + end) / 2, 340, 'Thermal\ndebinding\n(200-500 C)', ha='center', va='center',
                color=oven.PYROLYSIS_RANGE_COLOR, fontsize=11, fontweight='bold',
                bbox={'facecolor':'white', 'alpha':0.65, 'edgecolor':'none', 'pad':3})
    cu_intervals = oven.temperature_range_intervals(times, temps, *oven.COPPER_SINTERING_RANGE_C)[:1]
    if cu_intervals:
        start, end = cu_intervals[0]
        ax.text((start + end) / 2, 520, 'Cu sintering\n(750-1083 C)', ha='center', va='center',
                color='#C95B12', fontsize=11, fontweight='bold',
                bbox={'facecolor':'white', 'alpha':0.65, 'edgecolor':'none', 'pad':3})

    for step in timing['steps']:
        if step['ramp_h'] > 0:
            x = (step['ramp_start_h'] + step['ramp_end_h']) / 2
            y = np.interp(x, times, temps)
            ax.annotate(f"{step['ramp_rate_c_per_h']:g} C/h", (x, y), xytext=(0, -22),
                        textcoords='offset points', ha='center', fontsize=9,
                        bbox={'facecolor':'white', 'alpha':0.85, 'edgecolor':'none', 'pad':2})
        if step['hold_h'] > 0:
            x = (step['hold_start_h'] + step['hold_end_h']) / 2
            ax.annotate(f"{step['target_temperature_c']:g} C", (x, step['target_temperature_c']),
                        xytext=(0, 8), textcoords='offset points', ha='center', fontsize=9,
                        bbox={'facecolor':'white', 'alpha':0.85, 'edgecolor':'none', 'pad':2})

    handles = [Line2D([0], [0], color='#171717', lw=2.2, label='Oven temperature'),
               Patch(facecolor='none', edgecolor=oven.PYROLYSIS_RANGE_COLOR, hatch='//', label='Thermal debinding (200-500 C)'),
               Patch(facecolor='none', edgecolor=oven.COPPER_RANGE_COLOR, hatch='//', label='Cu sintering (750-1083 C)'),
               Line2D([0], [0], color='#C95B12', lw=1.7, label='Cu melting point (1083 C)'),
               Patch(facecolor='#75AADB', alpha=0.35, label='Cooling')]
    ax.legend(handles=handles, loc='upper left', bbox_to_anchor=(1.02, 0.98), frameon=False, fontsize=9)
    # The reference style uses the atmosphere heading rather than a cycle title.
    ax.set_xlabel('Elapsed time [h]')
    ax.set_ylabel('Temperature [C]')
    ax.grid(alpha=0.22)
    fig.patch.set_facecolor('white')
    if save:
        path = OUTPUT_DIR / f'{safe_name(name)}.png'
        fig.savefig(path, dpi=180, bbox_inches='tight')
        print(path)
    return fig, ax


In [4]:
# Generate one PNG for every saved thermal cycle.
for preset_name, preset_data in presets.items():
    fig, _ = plot_cycle(preset_name, preset_data, save=True)
    plt.close(fig)


c:\Users\sarklug\Documents\Visual Studio Code\Oven_Project\thermal_cycle_plots\test_n2.png
c:\Users\sarklug\Documents\Visual Studio Code\Oven_Project\thermal_cycle_plots\test_n1.png
c:\Users\sarklug\Documents\Visual Studio Code\Oven_Project\thermal_cycle_plots\test_n3.png
c:\Users\sarklug\Documents\Visual Studio Code\Oven_Project\thermal_cycle_plots\test_n4.png
c:\Users\sarklug\Documents\Visual Studio Code\Oven_Project\thermal_cycle_plots\Sintering_1.png
c:\Users\sarklug\Documents\Visual Studio Code\Oven_Project\thermal_cycle_plots\Sintering_2.png
c:\Users\sarklug\Documents\Visual Studio Code\Oven_Project\thermal_cycle_plots\Debinding_3.png
c:\Users\sarklug\Documents\Visual Studio Code\Oven_Project\thermal_cycle_plots\Sintering_3.png
c:\Users\sarklug\Documents\Visual Studio Code\Oven_Project\thermal_cycle_plots\Debinding___Sintering_1.png
c:\Users\sarklug\Documents\Visual Studio Code\Oven_Project\thermal_cycle_plots\Sintering_4.png
c:\Users\sarklug\Documents\Visual Studio Code\Oven_Pro